## LASSO Regression

## Learning Objectives

1. Use training and test datasets to evaluate predictive performance of a model
2. Demonstrate how to standardize predictors prior to LASSO regression
3. Utilize k-fold cross-validation to select the optimal $\alpha$ for LASSO regression models
4. Explain the difference between training, validation, and test datasets
5. Explain concepts such as overfitting and regularization
6. Identify and describe features of the LASSO objective function

## Review of Multiple Linear Regression

In multiple linear regression, we model a response variable $y$ using several predictor variables.

If we have predictors $x_1, x_2, \dots, x_p$, the model is

$$
\hat{y}
=
\beta_0
+
\beta_1 x_1
+
\beta_2 x_2
+
\dots
+
\beta_p x_p
$$

where:

- $\hat{y}$ is the predicted value of the response variable
- $\beta_0$ is the intercept
- $\beta_1, \beta_2, \dots, \beta_p$ are coefficients that measure how each predictor affects the prediction

The goal of regression is to choose coefficients that produce predictions close to the observed data.

In [1]:
import pandas as pd

df = pd.read_csv("../data/chem_data.csv")

df.head()

,Chem_1,Chem_2,Chem_3,Chem_4,Chem_5,Chem_6,Chem_7,Chem_8,Chem_9,Chem_10,...,Chem_92,Chem_93,Chem_94,Chem_95,Chem_96,Chem_97,Chem_98,Chem_99,Chem_100,Viscosity
0,-0.560476,0.787739,-0.715242,1.430402,1.074012,1.538430,-1.014114,0.331434,0.619850,-0.146427,...,0.117280,-0.685185,1.070008,1.623769,0.202615,-0.406072,1.516362,0.618266,0.298664,11.846806
1,-0.230177,0.769042,-0.752689,1.046629,-0.027347,-0.109710,-0.791314,-0.189847,-0.757510,-1.433562,...,2.001070,0.228735,-0.025951,0.173400,0.286926,-0.883253,0.384977,-0.231239,0.392163,2.845704
2,1.558708,0.332203,-0.938539,0.435289,-0.033330,0.511471,0.299594,0.470493,0.851525,-0.790608,...,0.345479,0.497824,-1.111060,-0.865529,0.760842,-0.378569,-0.624962,0.323101,0.247900,10.008446
3,0.070508,-1.008377,-1.052513,0.715178,-1.516068,0.213958,1.639052,-0.951680,-0.747930,0.885112,...,0.893642,1.881851,-1.617364,-0.562742,-1.289588,0.721854,0.999535,-0.586461,-2.798429,11.726238
4,0.129288,-0.119453,-0.437160,0.917175,0.790385,-0.186121,1.084617,1.157910,0.630240,0.903076,...,-0.008244,-0.401593,1.998328,-1.652575,0.946051,0.206937,0.103489,-0.266878,0.168881,-18.167885


## Measuring Prediction Error

For each observation:

- $y_i$ = actual value
- $\hat{y}_i$ = predicted value

The prediction error is

$$
y_i - \hat{y}_i
$$

To measure overall prediction quality, we use the **Mean Squared Error (MSE)**:

$$
\text{MSE}
=
\frac{1}{n}
\sum_{i=1}^{n}
(y_i - \hat{y}_i)^2
$$

where $n$ is the number of observations.

- Large errors are penalized heavily because the errors are squared
- Smaller MSE values indicate better predictions

## How Multiple Linear Regression Chooses Coefficients

Multiple linear regression chooses the coefficients

$$
\beta_0, \beta_1, \dots, \beta_p
$$

that minimize the Mean Squared Error on the training data.

In other words, regression searches for the line (or hyperplane) that makes the observed data points as close as possible to the predicted values on average.

This process is called **least squares estimation**.

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

y = df["Viscosity"]
X = df.drop(columns="Viscosity")

full_model = LinearRegression()
full_model.fit(X, y)
full_preds = full_model.predict(X)

print("RMSE: ", root_mean_squared_error(y, full_preds))

RMSE:  1.9507888062478473


## Train MSE, Test MSE, and Generalization

When we fit a regression model, we usually care about more than just how well the model fits the data we already have.

We also want the model to make good predictions on **new data** that it has never seen before.

To understand this, we distinguish between:

- **Training data**: the data used to fit the model
- **Test data**: new data that was not used when fitting the model

## Training Mean Squared Error

Suppose we have training observations

$$
(x_1, y_1), (x_2, y_2), \dots, (x_n, y_n)
$$

After fitting a model, we obtain predictions

$$
\hat{y}_1, \hat{y}_2, \dots, \hat{y}_n
$$

The **training Mean Squared Error (train MSE)** is

$$
\text{Train MSE}
=
\frac{1}{n}
\sum_{i=1}^{n}
(y_i - \hat{y}_i)^2
$$

This measures how well the model fits the training data.

A smaller train MSE means the model fits the training data more closely.

## Test Mean Squared Error

Now suppose we collect new observations that were **not used** to fit the model.

We denote these observations by

$$
(x_1^{\text{test}}, y_1^{\text{test}}),
\dots,
(x_m^{\text{test}}, y_m^{\text{test}})
$$

The model makes predictions

$$
\hat{y}_1^{\text{test}},
\dots,
\hat{y}_m^{\text{test}}
$$

The **test Mean Squared Error (test MSE)** is

$$
\text{Test MSE}
=
\frac{1}{m}
\sum_{i=1}^{m}
\left(
y_i^{\text{test}}
-
\hat{y}_i^{\text{test}}
\right)^2
$$

This measures how well the model predicts new observations.

## Splitting Data Into Training and Test Sets

To evaluate how well a model generalizes to new observations, we need to test it on data that were **not used** to fit the model.

For this reason, we often divide the data into:

- a **training set**
- a **test set**

## The Training Set

The **training set** is used to fit the model.

The model learns relationships between the predictors and the response variable using only the training data.

For example:

- estimating regression coefficients
- selecting patterns from the data
- fitting predictive relationships

all occur using the training set.

## The Test Set

The **test set** is used to evaluate predictive performance.

The model does **not** see the test data during training.

After fitting the model on the training set, we use the model to predict the response values in the test set.

We then compute metrics such as test MSE to measure how well the model performs on new observations.

## Why We Split the Data

If we evaluate the model using the same data that were used to fit it, we may get an overly optimistic estimate of performance.

This happens because the model has already adapted itself to those observations.

Using a separate test set gives a more honest estimate of how well the model generalizes.

## A Common Split

A common approach is:

- 70%–80% of the data for training
- 20%–30% of the data for testing

The exact proportions can vary depending on the size of the dataset.

## The General Workflow

A typical predictive modeling workflow looks like this:

1. Split the data into training and test sets
2. Fit the model using the training data
3. Use the fitted model to predict the test data
4. Compute test MSE (or another metric)
5. Compare models based on test performance

## Important Principle

The test set should represent **new, unseen data**.

To obtain a fair evaluation:

- the test set must not influence model fitting
- the test set should not be used to tune the model

Otherwise, information from the test set can leak into the model-building process and produce overly optimistic results.

Later, we will see that cross-validation provides a more efficient way to estimate predictive performance while still protecting against overfitting.

## Train and Test Splits Using Scikit-Learn

We can use the `train_test_split` method in the `model_selection` submodule of the scikit-learn library. You can provide the original feature matrix `X` and response vector `y` and then specify the percentage of data you would like to serve as a test dataset.

```python
sklearn.model_selection.train_test_split(*arrays, test_size=None, train_size=None, 
                                         random_state=None, shuffle=True, stratify=None)
```

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## The Difference Between Train MSE and Test MSE

- **Train MSE** measures performance on data the model already saw
- **Test MSE** measures performance on new, unseen data

A model can achieve extremely low train MSE simply by fitting the training data very closely.

However, this does **not** guarantee that the model will predict new observations well.

In [4]:
train_model = LinearRegression()

train_model.fit(X_train, y_train)

test_preds = train_model.predict(X_test)

print("RMSE: ", root_mean_squared_error(y_test, test_preds))

RMSE:  9.594960900480068


### Generalization

A model **generalizes well** if it performs well on new data, not just the training data.

In practice, our goal is usually:

- not just to minimize train MSE
- but to achieve **low test MSE**

This is important because real predictive tasks always involve future or unseen observations.

A model that memorizes the training data but performs poorly on new data is not useful.

## Overfitting

A model **overfits** when it learns the training data too closely, including random noise and accidental patterns that will not appear in future data.

An overfit model usually has:

- very low training error
- but poor performance on new observations

In other words, the model memorizes the training data instead of learning the underlying relationship.

## What Causes Overfitting?

Overfitting often happens when a model is too flexible relative to the amount of available data.

Common causes include:

- too many predictor variables
- models that are excessively flexible
- fitting noise in the data
- using a model that is more complicated than necessary

As model complexity increases:

- training MSE almost always decreases
- but test MSE may eventually increase

This happens because the model starts fitting random variation instead of meaningful signal.

## Bias-Variance Trade-Off

When building predictive models, there is usually a trade-off between:

- making the model too simple
- making the model too flexible

This is called the **bias-variance trade-off**.

Understanding this trade-off helps explain why overfitting happens and why we often prefer models that are not excessively complicated.

## High Bias

A model with **high bias** is too simple to capture the true relationship in the data.

Such a model tends to make systematic errors.

For example:

- fitting a straight line to a relationship that is strongly curved
- ignoring important predictors
- using a model that is too restrictive

High-bias models tend to:

- underfit the data
- have relatively high training error
- miss important patterns

## High Variance

A model with **high variance** is too sensitive to the particular training data.

Small changes in the training data can lead to large changes in the fitted model.

Highly flexible models often have high variance because they try to fit every detail of the training data, including random noise.

High-variance models tend to:

- achieve very low training error
- overfit the data
- perform poorly on new observations

## The Trade-Off

As model flexibility increases:

- bias generally decreases
- variance generally increases

A good predictive model balances these two effects.

We want a model that is:

- flexible enough to capture important patterns
- but not so flexible that it memorizes noise

The best model is often somewhere in the middle.

## Motivation for Regularization

We have seen that highly flexible models can overfit the training data.

As model complexity increases:

- training MSE usually decreases
- but test MSE may eventually increase

This happens because very flexible models can begin fitting random noise instead of meaningful signal.

To build models that generalize well, we need ways to control model complexity.

This is the motivation for **regularization**.



## What Is Regularization?

Regularization is a collection of techniques that discourage models from becoming excessively complex.

The main idea is:

> Instead of allowing the model coefficients to become arbitrarily large, we place restrictions on them.

Regularization encourages simpler models that are less sensitive to random fluctuations in the training data.

As a result, regularized models often:

- generalize better to new observations
- have lower test MSE
- are less prone to overfitting

## Regularization and the Bias-Variance Trade-Off

Recall the bias-variance trade-off:

- highly flexible models tend to have low bias but high variance
- very simple models tend to have high bias but low variance

Regularization helps balance these two effects.

By slightly restricting the model, we often:

- increase bias a little
- but reduce variance substantially

This trade-off can improve predictive performance on unseen data.

## The LASSO Objective Function

LASSO regression chooses coefficients that minimize

$$
\frac{1}{n}
\sum_{i=1}^{n}
(y_i - \hat{y}_i)^2
+
\alpha
\sum_{j=1}^{p}
|\beta_j|
$$

The first term is the usual Mean Squared Error.

The second term is called the **LASSO penalty**.

## The Penalty Term

The penalty is

$$
\sum_{j=1}^{p}
|\beta_j|
$$

which is the sum of the absolute values of the coefficients.

This term becomes large when the coefficients become large.

As a result, LASSO prefers models with:

- smaller coefficients
- less complexity
- greater stability

## The Tuning Parameter $\alpha$

The quantity $\alpha$ controls how strongly the penalty is applied.

If $\alpha = 0$, then there is no penalty at all.

In this case, LASSO becomes ordinary least squares regression.

As $\alpha$ increases:

- the penalty becomes more important
- coefficients are pushed closer to zero
- the model becomes simpler

Very large values of $\alpha$ can shrink nearly all coefficients toward zero.

## Why This Helps Prevent Overfitting

Large coefficients can make predictions highly sensitive to small fluctuations in the training data.

By discouraging large coefficients, LASSO reduces model flexibility and variance.

This often improves performance on new observations.

In other words, LASSO intentionally allows a little more bias in order to reduce variance and improve generalization.

## A Key Feature of LASSO

One particularly important property of LASSO is that it can force some coefficients to become **exactly zero**.

When this happens:

- the corresponding variables are effectively removed from the model

This means LASSO can automatically perform **variable selection** while also reducing overfitting.

## Sparsity and Automatic Variable Selection

One of the most important features of LASSO is that it can produce **sparse models**.

A sparse model is a model in which many coefficients are exactly zero.

When a coefficient is zero, the corresponding predictor has no effect on the predictions and is effectively removed from the model.

## Sparsity

Suppose we have a regression model with predictors

$$
x_1, x_2, \dots, x_p
$$

LASSO may produce coefficients such as

$$
\beta_1 = 2.1
$$

$$
\beta_2 = 0
$$

$$
\beta_3 = -1.4
$$

$$
\beta_4 = 0
$$

In this case:

- $x_1$ and $x_3$ remain in the model
- $x_2$ and $x_4$ are removed

The model is therefore simpler and more compact.

## Automatic Variable Selection

Because LASSO can set coefficients equal to zero automatically, it performs **variable selection** during model fitting.

This means we do not need to manually decide which variables to remove.

Instead, LASSO identifies predictors that appear less useful for prediction and shrinks their coefficients all the way to zero.

## Why Variable Selection Is Useful

Real datasets often contain:

- irrelevant predictors
- weak predictors
- redundant predictors
- noisy variables

Including too many unnecessary predictors can increase model complexity and contribute to overfitting.

By removing unimportant variables, LASSO can:

- reduce overfitting
- improve interpretability
- simplify the model
- improve prediction on new data

## Parsimonious Models

A model that achieves good predictive performance using relatively few variables is often called a **parsimonious model**.

Parsimonious models are desirable because they are:

- easier to understand
- easier to explain
- often more stable
- less likely to overfit

LASSO is widely used because it can automatically produce parsimonious models.

## Important Caveat

A coefficient being set to zero does **not** necessarily mean the variable is completely unrelated to the response.

LASSO removes variables based on predictive usefulness within the fitted model.

In particular:

- correlated predictors can compete with each other
- LASSO may keep one correlated variable while removing another

So variable selection results should be interpreted carefully.

## LASSO Regression Using Scikit-Learn

We can fit LASSO regression models in scikit-learn using the `Lasso` class in the `linear_model` submodule in scikit-learn. We can specify a value for $\alpha$ and it will find the coefficients that minimize the LASSO objective function.

```python
class sklearn.linear_model.Lasso(alpha=1.0, *, fit_intercept=True, precompute=False, copy_X=True, 
                                max_iter=1000, tol=0.0001, warm_start=False, positive=False, 
                                random_state=None, selection='cyclic')
```

In [5]:
from sklearn.linear_model import Lasso

lasso_model = Lasso(alpha=1)
lasso_model.fit(X_train, y_train)

lasso_preds = lasso_model.predict(X_test)

print("RMSE: ", root_mean_squared_error(y_test, lasso_preds))

RMSE:  4.451440240606939


## Feature Scaling and Standardization

Before fitting a LASSO model, it is usually important to **standardize** the predictor variables.

This means transforming the predictors so that they are all placed on a similar scale.

## Why Predictors Can Have Different Scales

Different variables may naturally have very different units and magnitudes.

For example:

- income might be measured in dollars
- age might be measured in years
- house size might be measured in square feet

As a result, some variables can have much larger numerical values than others.

## What Standardization Does

A common approach is to transform each predictor so that it has:

- mean equal to 0
- standard deviation equal to 1

For a predictor $x$, the standardized version is

$$
x_{\text{standardized}}
=
\frac{x - \text{mean}(x)}
{\text{standard deviation}(x)}
$$

After standardization:

- all predictors are centered around 0
- all predictors have comparable scales

## Why Scaling Matters for LASSO

Recall that the LASSO penalty is

$$
\alpha
\sum_{j=1}^{p}
|\beta_j|
$$

This penalty depends directly on the sizes of the coefficients.

If predictors are measured on very different scales, then the penalty affects them unevenly.

## An Intuition

Suppose we have two predictors:

- income measured in dollars
- age measured in years

A one-unit increase in income is tiny relative to the overall scale of the variable.

A one-unit increase in age is much larger relative to its scale.

Without standardization:

- coefficients are not directly comparable
- LASSO may penalize variables unfairly
- variable selection can become distorted

## What Could Go Wrong Without Scaling?

If predictors are not standardized:

- variables with large scales may receive artificially small coefficients
- variables with small scales may receive artificially large coefficients
- the penalty may favor some variables simply because of their units

This can lead to misleading results.

## Standardization Makes the Penalty Fair

After standardization:

- all predictors are measured on comparable scales
- the penalty treats coefficients more fairly
- coefficient sizes become easier to compare

As a result, variable selection and coefficient shrinkage behave more appropriately.

## Standardization of Variables in Scikit-Learn

We need to make sure that the we standardize the training data and then apply that same transformation to the test data. The `StandardScaler` class in the `preprocessing` submodule of scikit-learn makes this easy. It computes the mean and standard deviation for each variable in the training data and then lets us reuse these on the test data.

```python
class sklearn.preprocessing.StandardScaler(*, copy=True, with_mean=True, with_std=True)
```

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

scaled_model = Lasso(alpha=1)
scaled_model.fit(X_train_scaled, y_train)
scaled_preds = scaled_model.predict(X_test_scaled)

print("RMSE: ", root_mean_squared_error(y_test, scaled_preds))

RMSE:  4.457010415418506


## Why We Cannot Choose $\alpha$ Using Training Error Alone

Recall that the tuning parameter $\alpha$ controls the strength of the LASSO penalty.

Different values of $\alpha$ produce different models:

- small $\alpha$ values produce more flexible models
- large $\alpha$ values produce simpler models with more shrinkage

A natural question is:

> How should we choose $\alpha$?

It may seem reasonable to choose the value of $\alpha$ that produces the smallest training MSE.

However, this approach does not work well.

## What Happens to Training MSE?

As $\alpha$ becomes smaller:

- the penalty weakens
- the model becomes more flexible
- the model fits the training data more closely

In fact, the smallest training MSE usually occurs when:

$$
\alpha \approx 0
$$

which is very close to ordinary least squares regression.

## The Problem

Minimizing training MSE alone encourages overly flexible models.

But highly flexible models are more likely to overfit the training data.

As a result:

- training MSE may become very small
- while test MSE becomes larger

This means that the model generalizes poorly to new observations.

## Our Real Goal

In predictive modeling, we do not primarily care about minimizing training error.

Instead, we care about building models that perform well on unseen data.

That means we want to choose $\alpha$ so that:

- test MSE is small
- generalization is good
- overfitting is reduced

## The Trade-Off

As $\alpha$ changes:

- smaller $\alpha$ values reduce bias but increase variance
- larger $\alpha$ values increase bias but reduce variance

The best predictive performance usually occurs somewhere in the middle.

If $\alpha$ is:

- too small $\rightarrow$ overfitting
- too large $\rightarrow$ underfitting

So we need a principled way to estimate which value of $\alpha$ gives the best predictive performance on new data.

## Why We Need Cross-Validation

Ideally, we would like to know the test MSE for every possible value of $\alpha$.

But we do not usually have unlimited new test data available.

Cross-validation provides a practical way to estimate predictive performance using the available training data.

We will use cross-validation to choose the value of $\alpha$ that is expected to generalize best.

## Validation Sets and the Need for Cross-Validation

We have seen that we cannot choose the tuning parameter $\alpha$ using training MSE alone.

Training error tends to favor overly flexible models that overfit the data.

Instead, we want to estimate how well different values of $\alpha$ perform on new observations.

One approach is to use a **validation set**.

## The Basic Idea

We divide the available training data into two parts:

- a smaller training set
- a validation set

The process works as follows:

1. Fit the model using the smaller training set
2. Evaluate the model on the validation set
3. Compute the validation MSE
4. Repeat for different values of $\alpha$
5. Choose the value of $\alpha$ with the best validation performance

The validation set acts like "new" data that the model did not use during fitting.

## Why This Helps

A model that performs well on the validation set is more likely to generalize well to future observations.

Validation MSE gives a better estimate of predictive performance than training MSE because the model is evaluated on unseen data.

This helps us avoid choosing models that overfit.

## The Problem with a Single Validation Set

Although validation sets are useful, they have an important limitation.

The results can depend heavily on the particular random split of the data.

For example:

- one split may accidentally produce an "easy" validation set
- another split may produce a more difficult validation set

As a result:

- estimated performance can vary substantially
- the selected value of $\alpha$ may be unstable

This problem becomes worse when the dataset is small.

## Motivation for Cross-Validation

Cross-validation improves on the validation set approach.

Instead of relying on a single train-validation split, cross-validation repeatedly splits the data into different training and validation subsets.

This provides:

- a more stable estimate of predictive performance
- better use of the available data
- more reliable tuning parameter selection

Cross-validation is one of the most important tools in modern predictive modeling.

It is commonly used to choose tuning parameters such as the LASSO regularization strength $\alpha$.

## k-Fold Cross-Validation

k-fold cross-validation is a method for estimating how well a model will generalize to unseen data.

It is commonly used to:

- estimate test error
- compare models
- choose tuning parameters such as the LASSO penalty $\alpha$

## Core Idea

Rather than relying on a single train-validation split, we repeatedly train and evaluate the model using different parts of the dataset.

This produces a more reliable estimate of the model’s **generalization error**.

## The Procedure

Choose a value of $k$ (commonly $5$ or $10$).

Split the dataset into $k$ approximately equal-sized groups called **folds**.

Then repeat the following process $k$ times:

- hold out one fold as the validation set
- train the model on the remaining $k-1$ folds
- compute the validation error on the held-out fold

## Cross Validation Example

For example, when $k=5$:

| Iteration | Training Folds | Validation Fold |
|---|---|---|
| 1 | 2,3,4,5 | 1 |
| 2 | 1,3,4,5 | 2 |
| 3 | 1,2,4,5 | 3 |
| 4 | 1,2,3,5 | 4 |
| 5 | 1,2,3,4 | 5 |

This produces $k$ validation error estimates:

$$
\text{MSE}_1,\text{MSE}_2,\dots,\text{MSE}_k
$$


## Estimating Generalization Error

The cross-validation estimate of generalization error is the average of the validation errors across all folds:

$$
\text{CV Error}
=
\frac{1}{k}
\sum_{i=1}^{k}
\text{MSE}_i
$$

This quantity estimates how much prediction error we should expect on new, unseen data.

Because every observation is used for validation exactly once and for training $k-1$ times, the estimate is typically more stable than using a single validation split.


## Why k-Fold Cross-Validation Works Well

### Efficient Use of Data

Each observation serves two roles:

- training data in most iterations
- validation data once

This avoids permanently holding out a large validation set.

### More Reliable Error Estimates

Averaging across multiple train-validation splits reduces sensitivity to any one random split of the data.

As a result:

- model comparisons become more reliable
- tuning parameter selection becomes more stable

## Choosing $k$

Common choices are:

- $k=5$
- $k=10$

These usually provide a good tradeoff between:

- computational cost
- reliability of the estimated test error


## Cross-Validation for LASSO

For LASSO regression, we repeat the entire k-fold procedure for many values of $\alpha$.

For each $\alpha$:

1. fit the model across all folds
2. compute the average validation MSE
3. calculate the cross-validation error

We then choose the value of $\alpha$ that minimizes the estimated generalization error.

## Train, Validation, and Test Sets

When building predictive models, it is important to separate the roles of:

- model fitting
- model selection
- final model evaluation

To do this properly, we often divide the data into three parts:

- a training set
- a validation set
- a test set

Each dataset serves a different purpose.

## The Training Set

The **training set** is used to fit the model.

This is where the model learns relationships between the predictors and the response variable.

For example:

- estimating regression coefficients
- fitting a LASSO model
- learning patterns from the data

all occur using the training set.

## The Validation Set

The **validation set** is used for model selection.

For example, we may use it to:

- choose the value of $\alpha$
- compare competing models
- estimate predictive performance during model development

The validation set helps us decide which model appears to generalize best.

Importantly:

- the validation data are not used directly to fit the model coefficients

## The Test Set

The **test set** is used only at the very end of the modeling process.

Its purpose is to provide a final, unbiased estimate of predictive performance on completely unseen data.

The test set should not be used:

- to fit the model
- to choose tuning parameters
- to compare many candidate models during development

Otherwise, information from the test set can leak into the modeling process.

## Why We Need All Three

Each dataset answers a different question.

| Dataset | Purpose |
|---|---|
| Training set | Fit the model |
| Validation set | Select models and tuning parameters |
| Test set | Final evaluation of generalization |

Keeping these roles separate helps prevent overly optimistic performance estimates.

## Where Cross-Validation Fits In

In practice, cross-validation is often used instead of a single validation set.

A common workflow is:

1. Split the data into:
   - training data
   - test data

2. Use cross-validation within the training data to:
   - choose $\alpha$
   - compare models

3. Fit the final model using the full training data

4. Evaluate the final model once on the test data

This is the workflow we will usually follow.

## Why the Test Set Must Be Protected

Suppose we repeatedly adjust the model after viewing test set performance.

Eventually, the model may become specialized to the test set itself.

This would make the test performance artificially optimistic.

The test set should therefore remain untouched until all model decisions are complete.

## Cross-Validation for LASSO in Scikit-Learn

Cross-validation for LASSO regression is made easy using the `LassoCV` class. We only need to specify the number of folds for cross validation using the `cv` argument and it will use cross-validation to look for the value of $\alpha$ that minimizes the validation error and the LASSO regression model using that $\alpha$.

```python
class sklearn.linear_model.LassoCV(*, eps=0.001, n_alphas='deprecated', alphas='warn', 
                                   fit_intercept=True, precompute='auto', max_iter=1000, tol=0.0001, 
                                   copy_X=True, cv=None, verbose=False, n_jobs=None, positive=False, 
                                   random_state=None, selection='cyclic')
```

In [7]:
from sklearn.linear_model import LassoCV

lasso_CV_model = LassoCV(cv=10)
lasso_CV_model.fit(X_train_scaled, y_train)

lasso_CV_preds = lasso_CV_model.predict(X_test_scaled)

print("Selected alpha: ", lasso_CV_model.alpha_)
print("RMSE: ", root_mean_squared_error(y_test, lasso_CV_preds))

Selected alpha:  0.3274826805238531
RMSE:  4.16763074229008


## Seeing Active Predictors in LASSO Model

We can look at the `coef_` attribute of the fitted LASSO model to see which coefficients are active.

In [8]:
{X.columns[i] : lasso_CV_model.coef_[i] for i in range(len(X.columns))}

{'Chem_1': 3.3471159650127054,
 'Chem_2': -0.0,
 'Chem_3': -0.0,
 'Chem_4': -0.040819485032487834,
 'Chem_5': -0.0,
 'Chem_6': 0.0,
 'Chem_7': -0.0,
 'Chem_8': 0.0,
 'Chem_9': -0.0,
 'Chem_10': -0.0,
 'Chem_11': 0.0,
 'Chem_12': -0.0,
 'Chem_13': -0.0,
 'Chem_14': 0.0,
 'Chem_15': -0.0,
 'Chem_16': 0.0,
 'Chem_17': 0.0,
 'Chem_18': 0.0,
 'Chem_19': -0.0,
 'Chem_20': 0.0,
 'Chem_21': -2.4080074206847213,
 'Chem_22': 0.0,
 'Chem_23': 0.0,
 'Chem_24': -0.0,
 'Chem_25': 0.0,
 'Chem_26': -0.0,
 'Chem_27': 0.0,
 'Chem_28': -0.0,
 'Chem_29': 0.0,
 'Chem_30': -0.1619231596528599,
 'Chem_31': -0.020062539273975354,
 'Chem_32': 0.0,
 'Chem_33': 0.0,
 'Chem_34': -0.0,
 'Chem_35': -0.017592730625381074,
 'Chem_36': -0.0,
 'Chem_37': 0.0,
 'Chem_38': -0.0,
 'Chem_39': -0.0,
 'Chem_40': 0.22079417242854968,
 'Chem_41': 0.0,
 'Chem_42': -0.0,
 'Chem_43': 0.0,
 'Chem_44': 2.6369099522435113,
 'Chem_45': -0.09273285369776481,
 'Chem_46': -0.0,
 'Chem_47': -0.0,
 'Chem_48': -0.0,
 'Chem_49': 0.0,
 'Chem